In [ ]:
import ast
import json
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics import (accuracy_score,f1_score,precision_score,recall_score)
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm
from google.colab import drive
from google.colab import userdata


In [ ]:
SEED = 42
PILOT_SIZE = 30
DEV_SIZE = 10
ANNOTATION_VALIDATION_SIZE = 20
MAX_RETRIES = 3

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "SCOTBESS_CLEAN.csv"
DEFINITIONS_PATH = (PROJECT_DIR / "SCOTBESS_labels_definitions_v2.xlsx")
PILOT_PATH = (PROJECT_DIR / "SCOTBESS_pilot_manual.xlsx")
PILOT_LABELED_PATH = (PROJECT_DIR / "SCOTBESS_pilot_manual_labeled_v2.xlsx")

PILOT_SPLIT_PATH = (PROJECT_DIR / "SCOTBESS_pilot_manual_labeled_with_split.xlsx")
DEVELOPMENT_PATH = (PROJECT_DIR / "SCOTBESS_pilot_development_10.xlsx")
ANNOTATION_VALIDATION_PATH = (PROJECT_DIR / "SCOTBESS_annotation_validation_20.xlsx")
DEV_PREDICTIONS_PATH = (PROJECT_DIR / "SCOTBESS_development_predictions.csv")
DEV_SUMMARY_PATH = (
PROJECT_DIR / "SCOTBESS_development_model_comparison.csv")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#labels
labels_df = pd.read_excel(DEFINITIONS_PATH)

required_definition_columns = {"label", "definition"}
missing = required_definition_columns - set(labels_df.columns)
if missing:
    raise ValueError(
        f"Definitions file is missing columns: {sorted(missing)}")

labels_df["label"] = labels_df["label"].astype(str).str.strip()
labels_df["definition"] = (labels_df["definition"].astype(str).str.strip())

if labels_df["label"].duplicated().any():
    raise ValueError("Duplicate labels found in definitions file.")

label_descriptions = dict(
    zip(labels_df["label"], labels_df["definition"]))

LABELS = list(label_descriptions)
LABEL_SET = set(LABELS)
LABEL_ORDER = {label: position for position, label in enumerate(LABELS)}

print(f"Labels: {len(LABELS)}")


Labels: 20


In [ ]:
#dataset
df = pd.read_csv(DATA_PATH)

required_columns = {"document_id" "project", "source", "final_masked_text"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(
        f"Dataset is missing columns: {sorted(missing)}")

if df["document_id"].duplicated().any():
    raise ValueError("document_id must be unique.")

if df["final_masked_text"].isna().any():
    raise ValueError(
        "final_masked_text contains missing values.")

df["final_masked_text"] = (df["final_masked_text"].astype(str).str.strip())

if df["final_masked_text"].eq("").any():
    raise ValueError(
        "final_masked_text contains blank responses.")

print(f"Responses: {len(df):}")


Responses: 1,675


## Creation and validation of the manual pilot sample



In [ ]:
def create_pilot_sample(data: pd.DataFrame, n: int, seed: int):
    if n > len(data):
        raise ValueError("PILOT_SIZE exceeds the number of dataset rows.")

    work = data.copy()
    work["text_characters"] = work["final_masked_text"].str.len()
    work["length_bin"] = pd.qcut(work["text_characters"], q=5, labels=False, duplicates="drop")
    work["stratum"] = work["source"].astype(str) + "__length_" + work["length_bin"].astype(str)

    counts = work["stratum"].value_counts()

    if len(counts) < n and counts.min() >= 2:
        allocation = np.floor(counts / counts.sum() * n).astype(int).clip(lower=1)

        while allocation.sum() > n:
            candidates = allocation[allocation > 1]
            allocation.loc[candidates.idxmax()] -= 1

        while allocation.sum() < n:
            remaining = counts - allocation
            allocation.loc[remaining.idxmax()] += 1

        parts = []
        for stratum, take_n in allocation.items():
            group = work[work["stratum"] == stratum]
            parts.append(group.sample(n=int(take_n), random_state=seed))

        pilot_sample = pd.concat(parts, ignore_index=True)

    else:
        pilot_sample = work.sample(n=n, random_state=seed)

    pilot_sample = pilot_sample.sample(frac=1, random_state=seed).reset_index(drop=True)
    pilot_sample.insert(0, "pilot_id", np.arange(1, len(pilot_sample) + 1))
    pilot_sample["manual_gold_labels"] = ""

    output_columns = [
        "pilot_id",
        "document_id",
        "project",
        "filename",
        "source",
        "text_characters",
        "final_masked_text",
        "manual_gold_labels"]

    return pilot_sample[[column for column in output_columns if column in pilot_sample.columns]]


if not PILOT_PATH.exists():
    pilot_to_label = create_pilot_sample(df, PILOT_SIZE, SEED)
    pilot_to_label.to_excel(PILOT_PATH, index=False)
    print(f"Created: {PILOT_PATH}")
else:
    print(f"Pilot already exists and was not overwritten: {PILOT_PATH}")

Pilot already exists and was not overwritten: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_pilot_manual.xlsx


In [ ]:
def parse_label_list(value) -> list[str]:
    if isinstance(value, list):
        values = value
    elif pd.isna(value) or not str(value).strip():
        raise ValueError("Blank label cell.")
    else:
        text = str(value).strip()
        try:
            values = json.loads(text)
        except json.JSONDecodeError:
            try:
                values = ast.literal_eval(text)
            except (ValueError, SyntaxError) as exc:
                raise ValueError(f"Could not parse labels: {value!r}") from exc

    if not isinstance(values, list):
        raise ValueError(f"Expected a list, received: {value!r}")

    if not all(isinstance(label, str) for label in values):
        raise ValueError(f"Every label must be a string: {value!r}")

    values = [label.strip() for label in values]

    unknown = sorted(set(values) - LABEL_SET)
    if unknown:
        raise ValueError(f"Unknown labels: {unknown}")

    return sorted(set(values), key=LABEL_ORDER.get)


assert PILOT_LABELED_PATH.exists()

pilot = pd.read_excel(PILOT_LABELED_PATH)

required_columns = {"pilot_id", "document_id", "manual_gold_labels"}
missing = required_columns - set(pilot.columns)

if missing:
    raise ValueError(f"Manual annotation file is missing columns: {sorted(missing)}")

if len(pilot) != PILOT_SIZE:
    raise ValueError(f"Expected {PILOT_SIZE} rows, found {len(pilot)}.")

if pilot["pilot_id"].duplicated().any():
    raise ValueError("Duplicate pilot_id values found.")

if pilot["document_id"].duplicated().any():
    raise ValueError("Duplicate document_id values found.")

pilot["gold_labels"] = pilot["manual_gold_labels"].apply(parse_label_list)

print(f"Validated {len(pilot)} manually annotated responses.")

Validated 30 manually annotated responses.


## Splitting the manual sample

10 for development, 20 for evaluation

In [ ]:
pilot = pilot.copy()

development_set = pilot.sample(n=DEV_SIZE, random_state=SEED)
annotation_validation_set = pilot.drop(development_set.index)

pilot["split"] = "annotation_validation"
pilot.loc[development_set.index, "split"] = "development"

development_set = development_set.sort_values("pilot_id").reset_index(drop=True)
annotation_validation_set = annotation_validation_set.sort_values("pilot_id").reset_index(drop=True)

assert len(development_set) == DEV_SIZE
assert len(annotation_validation_set) == ANNOTATION_VALIDATION_SIZE

In [ ]:
development_set.to_excel(DEVELOPMENT_PATH, index=False)

annotation_validation_set.to_excel(ANNOTATION_VALIDATION_PATH, index=False)

pilot.to_excel(PILOT_SPLIT_PATH, index=False)

## Annotation prompt


In [ ]:
definitions_block = "\n".join(f"{i + 1}. {row.label}: {row.definition}"for i, row in labels_df.iterrows())

ANNOTATION_INSTRUCTIONS = f"""You are an expert in annotating Scottish planning representations concerning battery energy storage system developments.

Assign every label supported by the response. This is a multi-label classification task. If none of the labels is supported, return an empty labels list.

Use the provided label definitions as the sole basis for assigning labels. Follow their inclusion criteria, exclusions, and boundaries carefully.

Rules:
- Use only exact label names from the permitted label set.
- Do not infer concerns, opinions, causes, or consequences that are not expressed.
- A single sentence or clause is sufficient when it clearly supports a label.
- Do not assign a label solely because a related topic or keyword is mentioned incidentally.
- Return only the JSON object, with no explanation or extra text.

PERMITTED LABELS AND DEFINITIONS
{definitions_block}
"""

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "description": "All and only the substantively supported labels.",
            "items": {"type": "string", "enum": LABELS},
        }
    },
    "required": ["labels"],
    "additionalProperties": False}



## API configurations


In [ ]:
api_key = userdata.get("open")
client = OpenAI(api_key=api_key)

CONFIGS = [
    {"config": "luna_none", "model": "gpt-5.6-luna", "effort": "none"},
    {"config": "terra_none", "model": "gpt-5.6-terra", "effort": "none"},
    {"config": "terra_low", "model": "gpt-5.6-terra", "effort": "low"}]

for model_id in sorted({config["model"] for config in CONFIGS}):
    try:
        client.models.retrieve(model_id)
        print(f"Available: {model_id}")
    except Exception as exc:
        raise ValueError(f"Model is unavailable to this API key: {model_id}") from exc

Available: gpt-5.6-luna
Available: gpt-5.6-terra


## Run the three configurations on the 10 development cases

In [ ]:
def classify_one(row: dict, config: dict) -> dict:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=config["model"],
                reasoning={"effort": config["effort"]},
                instructions=ANNOTATION_INSTRUCTIONS,
                input=f"Classify the following consultation response.\n\nRESPONSE:\n{row['final_masked_text']}",
                text={"format": {
                    "type": "json_schema",
                    "name": "scotbess_labels",
                    "strict": True,
                    "schema": LABEL_SCHEMA,
                }},
                max_output_tokens=500,
                store=False,
            )

            if response.status != "completed":
                raise RuntimeError(f"Response status: {response.status}")

            parsed = json.loads(response.output_text)
            labels = parse_label_list(parsed["labels"])

            return {
                "pilot_id": row["pilot_id"],
                "document_id": row["document_id"],
                "config": config["config"],
                "model": config["model"],
                "reasoning_effort": config["effort"],
                "predicted_labels": json.dumps(labels, ensure_ascii=False),
                "error": "",
            }

        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))

    return {
        "pilot_id": row["pilot_id"],
        "document_id": row["document_id"],
        "config": config["config"],
        "model": config["model"],
        "reasoning_effort": config["effort"],
        "predicted_labels": "",
        "error": repr(last_error),
    }


if DEV_PREDICTIONS_PATH.exists():
    predictions = pd.read_csv(DEV_PREDICTIONS_PATH)
else:
    predictions = pd.DataFrame()

completed = set()

if not predictions.empty:
    successful = predictions[predictions["error"].fillna("").eq("")]
    completed = set(zip(successful["pilot_id"], successful["config"]))

jobs = [
    (row, config)
    for config in CONFIGS
    for row in development_set.to_dict("records")
    if (row["pilot_id"], config["config"]) not in completed
]

print(f"API calls remaining: {len(jobs)}")

new_results = []

for row, config in tqdm(jobs):
    result = classify_one(row, config)
    new_results.append(result)

    predictions = pd.concat([predictions, pd.DataFrame([result])], ignore_index=True)
    predictions = predictions.drop_duplicates(subset=["pilot_id", "config"], keep="last").sort_values(["config", "pilot_id"])
    predictions.to_csv(DEV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

errors = predictions[~predictions["error"].fillna("").eq("")]

print(f"Saved: {DEV_PREDICTIONS_PATH}")
print(f"Failed calls: {len(errors)}")

if len(errors):
    display(errors[["pilot_id", "config", "error"]])

API calls remaining: 30


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_development_predictions.csv
Failed calls: 0


## Comparing the configurations



In [ ]:
mlb = MultiLabelBinarizer(classes=LABELS)
mlb.fit([LABELS])

MultiLabelBinarizer(classes=['Wildlife and Ecology', 'Traffic',
                             'Fire and Explosion Risk',
                             'Landscape, Visual and Heritage Impact',
                             'Consultation, Transparency and Information',
                             'Emergency Planning and Response',
                             'Water and Soil Contamination',
                             'Cumulative Impact', 'Light Pollution', 'Noise',
                             'Health and Wellbeing', 'Property Value',
                             'Planning Policy and Regulatory Compliance',
                             'Site Selection',
                             'Community and Economic Benefits',
                             'Decommissioning and Site Restoration',
                             'Project Need', 'Agricultural Land',
                             'Residential Proximity and Separation Distance',
                             'Grid Connection and Electrical Infrastructure'])

In [ ]:
successful_predictions = predictions[predictions["error"].fillna("").eq("")].copy()

expected = len(development_set) * len(CONFIGS)

if len(successful_predictions) != expected:
    raise ValueError(f"Expected {expected} successful predictions, found {len(successful_predictions)}.")

successful_predictions["predicted_label_list"] = successful_predictions["predicted_labels"].apply(parse_label_list)

comparison = successful_predictions.merge(
    development_set[["pilot_id", "document_id", "final_masked_text", "gold_labels"]],
    on=["pilot_id", "document_id"],
    how="inner",
    validate="many_to_one")

metric_rows = []

for config in CONFIGS:
    subset = comparison[comparison["config"] == config["config"]].copy()

    y_true = mlb.transform(subset["gold_labels"])
    y_pred = mlb.transform(subset["predicted_label_list"])

    metric_rows.append({
        "config": config["config"],
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "sample_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred)})

development_summary = pd.DataFrame(metric_rows).sort_values(
    ["micro_f1", "sample_f1"],
    ascending=False).reset_index(drop=True)

development_summary.to_csv(DEV_SUMMARY_PATH, index=False, encoding="utf-8-sig")

display(development_summary.style.format({
    "micro_precision": "{:.3f}",
    "micro_recall": "{:.3f}",
    "micro_f1": "{:.3f}",
    "sample_f1": "{:.3f}",
    "exact_match": "{:.3f}"}))

,config,micro_precision,micro_recall,micro_f1,sample_f1,exact_match
0,terra_low,0.982,1.000,0.991,0.994,0.900
1,terra_none,0.915,0.964,0.939,0.950,0.600
2,luna_none,0.930,0.946,0.938,0.938,0.500


## Inspecting mistakes before choosing the model


In [ ]:
def label_difference(row):
    gold = set(row["gold_labels"])
    predicted = set(row["predicted_label_list"])

    return pd.Series({"missing_labels": sorted(gold - predicted), "extra_labels": sorted(predicted - gold)})


error_details = comparison.join(
    comparison.apply(
        label_difference,
        axis=1))

mistakes = error_details[
    error_details["missing_labels"].map(bool)
    | error_details["extra_labels"].map(bool)][[
        "pilot_id",
        "config",
        "gold_labels",
        "predicted_label_list",
        "missing_labels",
        "extra_labels",
        "final_masked_text",
    ]].sort_values(["pilot_id", "config"])

display(mistakes)


,pilot_id,config,gold_labels,predicted_label_list,missing_labels,extra_labels,final_masked_text
1,9,luna_none,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Landscape, Visual an...",[],[Residential Proximity and Separation Distance],This formal objection is lodged in response to...
4,16,luna_none,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[Cumulative Impact],[Residential Proximity and Separation Distance],This proposal raises significant concerns rega...
14,16,terra_low,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Consultatio...",[],"[Consultation, Transparency and Information]",This proposal raises significant concerns rega...
24,16,terra_none,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[Cumulative Impact],[Grid Connection and Electrical Infrastructure...,This proposal raises significant concerns rega...
5,18,luna_none,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[Site Selection],[],Failure to Inform the Public on Safety Measure...
25,18,terra_none,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[Site Selection],[Agricultural Land],Failure to Inform the Public on Safety Measure...
7,25,luna_none,"[Traffic, Landscape, Visual and Heritage Impac...","[Traffic, Landscape, Visual and Heritage Impac...",[],[Agricultural Land],I feel this type of development is inappropria...
27,25,terra_none,"[Traffic, Landscape, Visual and Heritage Impac...","[Traffic, Landscape, Visual and Heritage Impac...",[],[Agricultural Land],I feel this type of development is inappropria...
8,28,luna_none,"[Traffic, Fire and Explosion Risk, Emergency P...","[Fire and Explosion Risk, Emergency Planning a...",[Traffic],[Water and Soil Contamination],I write to formally object to the proposal to ...
28,28,terra_none,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[],[Water and Soil Contamination],I write to formally object to the proposal to ...


##Performing the same check for Mistral models

In [ ]:
!pip install -q -U "mistralai>=2"


In [ ]:
from google.colab import userdata
from mistralai.client import Mistral
import time
from mistralai.client.models import (ResponseFormat, JSONSchema, TextChunk)
from typing import Literal
from pydantic import BaseModel, ConfigDict
from mistralai.client.models import TextChunk


In [ ]:
AllowedLabel = Literal[tuple(LABELS)]

class ScotBESSLabels(BaseModel):
    labels: list[AllowedLabel]

    model_config = ConfigDict(extra="forbid")

In [ ]:
MISTRAL_RESPONSE_FORMAT = ResponseFormat(
    type="json_schema",
    json_schema=JSONSchema(
        name="scotbess_labels",
        schema_definition=ScotBESSLabels.model_json_schema(),
        strict=True),)

In [ ]:
mistral_api_key = userdata.get("mistral")
mistral_client = Mistral(api_key=mistral_api_key)

In [ ]:
MISTRAL_CONFIGS = [
    {
        "config": "mistral_large_3",
        "model": "mistral-large-2512",
        "effort": None},
    {
        "config": "mistral_medium_3_5_none",
        "model": "mistral-medium-3-5",
        "effort": "none"},
    {
        "config": "mistral_medium_3_5_high", #testing with high reasoning as mistral only allows none or high, so there's no direct equivalent to gpt terra "low"
        "model": "mistral-medium-3-5",
        "effort": "high"}]

In [ ]:
def classify_one_mistral(row: dict, config: dict) -> dict:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            kwargs = {
                "model": config["model"],
                "messages": [
                    {"role": "system", "content": ANNOTATION_INSTRUCTIONS},
                    {
                        "role": "user",
                        "content": f"Classify the following consultation response.\n\nRESPONSE:\n{row['final_masked_text']}",
                    },
                ],
                "response_format": MISTRAL_RESPONSE_FORMAT,
                "max_tokens": 8000,
                "temperature": 0,
                "top_p": 1.0,
                "random_seed": SEED}

            if config["effort"] is not None:
                kwargs["reasoning_effort"] = config["effort"]

            response = mistral_client.chat.complete(**kwargs)
            content = response.choices[0].message.content

            if isinstance(content, str):
                final_text = content
            else:
                final_text = "".join(chunk.text for chunk in content if isinstance(chunk, TextChunk))

            if not final_text:
                raise ValueError("No final TextChunk returned.")

            parsed = ScotBESSLabels.model_validate_json(final_text)
            labels = parse_label_list(parsed.labels)

            return {
                "pilot_id": row["pilot_id"],
                "document_id": row["document_id"],
                "config": config["config"],
                "model": config["model"],
                "reasoning_effort": config["effort"],
                "predicted_labels": json.dumps(labels, ensure_ascii=False),
                "error": ""}

        except Exception as exc:
            last_error = exc

            if attempt < MAX_RETRIES:
                if "429" in str(exc):
                    time.sleep(20)
                else:
                    time.sleep(2 ** (attempt - 1))

    return {
        "pilot_id": row["pilot_id"],
        "document_id": row["document_id"],
        "config": config["config"],
        "model": config["model"],
        "reasoning_effort": config["effort"],
        "predicted_labels": "",
        "error": repr(last_error)}

In [ ]:
MISTRAL_DEV_PREDICTIONS_PATH = (PROJECT_DIR / "SCOTBESS_development_predictions_mistral.csv")

In [ ]:
if MISTRAL_DEV_PREDICTIONS_PATH.exists():
    mistral_predictions = pd.read_csv(MISTRAL_DEV_PREDICTIONS_PATH)
else:
    mistral_predictions = pd.DataFrame()

completed = set()

if not mistral_predictions.empty:
    successful = mistral_predictions[mistral_predictions["error"].fillna("").eq("")]
    completed = set(zip(successful["pilot_id"], successful["config"]))

jobs = [
    (row, config)
    for config in MISTRAL_CONFIGS
    for row in development_set.to_dict("records")
    if (row["pilot_id"], config["config"]) not in completed]

print(f"Mistral API calls remaining: {len(jobs)}")

for row, config in tqdm(jobs):
    result = classify_one_mistral(row, config)

    mistral_predictions = pd.concat([mistral_predictions, pd.DataFrame([result])], ignore_index=True)
    mistral_predictions = mistral_predictions.drop_duplicates(subset=["pilot_id", "config"], keep="last").sort_values(["config", "pilot_id"])

    mistral_predictions.to_csv(MISTRAL_DEV_PREDICTIONS_PATH, index=False, encoding="utf-8")

    #free tier limit is max 4 requests/minute
    time.sleep(16)

errors = mistral_predictions[~mistral_predictions["error"].fillna("").eq("")]

print(f"Saved: {MISTRAL_DEV_PREDICTIONS_PATH}")
print(f"Failed calls: {len(errors)}")

if len(errors):
    display(errors[["pilot_id", "config", "error"]])

Mistral API calls remaining: 26


  0%|          | 0/26 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_development_predictions_mistral.csv
Failed calls: 0


In [ ]:
successful_predictions = mistral_predictions[mistral_predictions["error"].fillna("").eq("")].copy()

expected = len(development_set) * len(MISTRAL_CONFIGS)

if len(successful_predictions) != expected:
    raise ValueError(f"Expected {expected} successful predictions, found {len(successful_predictions)}.")

successful_predictions["predicted_label_list"] = successful_predictions["predicted_labels"].apply(parse_label_list)

comparison = successful_predictions.merge(
    development_set[["pilot_id", "document_id", "final_masked_text", "gold_labels"]],
    on=["pilot_id", "document_id"],
    how="inner",
    validate="many_to_one")

metric_rows = []

for config in MISTRAL_CONFIGS:
    subset = comparison[comparison["config"] == config["config"]].copy()

    y_true = mlb.transform(subset["gold_labels"])
    y_pred = mlb.transform(subset["predicted_label_list"])

    metric_rows.append({
        "config": config["config"],
        "model": config["model"],
        "reasoning_effort": config["effort"],
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "sample_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred)})

mistral_development_summary = pd.DataFrame(metric_rows).sort_values(
    ["micro_f1", "sample_f1"],
    ascending=False).reset_index(drop=True)

MISTRAL_DEV_SUMMARY_PATH = PROJECT_DIR / "SCOTBESS_development_model_comparison_mistral.csv"

mistral_development_summary.to_csv(MISTRAL_DEV_SUMMARY_PATH, index=False, encoding="utf-8")

display(mistral_development_summary.style.format({
    "micro_precision": "{:.3f}",
    "micro_recall": "{:.3f}",
    "micro_f1": "{:.3f}",
    "sample_f1": "{:.3f}",
    "exact_match": "{:.3f}"}))

,config,model,reasoning_effort,micro_precision,micro_recall,micro_f1,sample_f1,exact_match
0,mistral_medium_3_5_high,mistral-medium-3-5,high,0.964,0.946,0.955,0.963,0.700
1,mistral_large_3,mistral-large-2512,None,0.962,0.911,0.936,0.947,0.500
2,mistral_medium_3_5_none,mistral-medium-3-5,none,0.885,0.964,0.923,0.915,0.400


In [ ]:
mistral_comparison = successful_predictions.merge(
    development_set[["pilot_id", "document_id", "final_masked_text", "gold_labels"]],
    on=["pilot_id", "document_id"],
    how="inner",
    validate="many_to_one")

In [ ]:
mistral_error_details = mistral_comparison.join(
    mistral_comparison.apply(
        label_difference,
        axis=1))


mistral_mistakes = mistral_error_details[
    mistral_error_details["missing_labels"].map(bool)
    | mistral_error_details["extra_labels"].map(bool)][
    [
        "pilot_id",
        "config",
        "gold_labels",
        "predicted_label_list",
        "missing_labels",
        "extra_labels",
        "final_masked_text"]].sort_values(["pilot_id", "config"])


display(mistral_mistakes)

,pilot_id,config,gold_labels,predicted_label_list,missing_labels,extra_labels,final_masked_text
1,9,mistral_large_3,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Landscape, Visual an...",[Health and Wellbeing],[],This formal objection is lodged in response to...
21,9,mistral_medium_3_5_none,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Landscape, Visual an...",[],[Residential Proximity and Separation Distance],This formal objection is lodged in response to...
2,10,mistral_large_3,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Emergency Planning a...","[Landscape, Visual and Heritage Impact, Planni...",[],"The need for battery storage facilities, such ..."
12,10,mistral_medium_3_5_high,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Landscape, Visual an...",[Planning Policy and Regulatory Compliance],[],"The need for battery storage facilities, such ..."
22,10,mistral_medium_3_5_none,"[Fire and Explosion Risk, Landscape, Visual an...","[Fire and Explosion Risk, Emergency Planning a...","[Landscape, Visual and Heritage Impact]",[],"The need for battery storage facilities, such ..."
4,16,mistral_large_3,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[],[Residential Proximity and Separation Distance],This proposal raises significant concerns rega...
14,16,mistral_medium_3_5_high,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Consultatio...",[Planning Policy and Regulatory Compliance],"[Consultation, Transparency and Information, R...",This proposal raises significant concerns rega...
24,16,mistral_medium_3_5_none,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[],[Residential Proximity and Separation Distance],This proposal raises significant concerns rega...
5,18,mistral_large_3,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[Residential Proximity and Separation Distance],[],Failure to Inform the Public on Safety Measure...
25,18,mistral_medium_3_5_none,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[],"[Agricultural Land, Grid Connection and Electr...",Failure to Inform the Public on Safety Measure...


In [ ]:
#Overall, Terra with low reasoning enabled performed best. Mistral 3.5 with reasoning performed comparably, but sligthly worse + it is much more expensive due to reasoning on "high" mode
#therefore Terra with "low" reasoning will be used as the final annotation model

In [ ]:
FINAL_CONFIG = next(config for config in CONFIGS if config["config"] == "terra_low")
VALIDATION_PREDICTIONS_PATH = (PROJECT_DIR / "SCOTBESS_annotation_validation_terra_low.csv")

print(FINAL_CONFIG)


{'config': 'terra_low', 'model': 'gpt-5.6-terra', 'effort': 'low'}


##TERRA-LOW STABILITY CHECK

In [ ]:
#3 independent inference runs on the 10 development documents
STABILITY_RUNS = 3
STABILITY_PATH = PROJECT_DIR / "SCOTBESS_terra_low_stability_3runs.csv"

#load previous progress if the cell/notebook was interrupted
if STABILITY_PATH.exists():
    stability_predictions = pd.read_csv(STABILITY_PATH)
else:
    stability_predictions = pd.DataFrame()

for run_id in range(1, STABILITY_RUNS + 1):
    print(f"\n=== Stability run {run_id}/{STABILITY_RUNS} ===")

    completed = set()

    if not stability_predictions.empty:
        existing = stability_predictions[
            (stability_predictions["run_id"] == run_id)
            & stability_predictions["error"].fillna("").eq("")]
        completed = set(existing["pilot_id"])

    jobs = [row for row in development_set.to_dict("records") if row["pilot_id"] not in completed]

    for row in tqdm(jobs):
        # classify_one() makes a fresh Terra API request here
        result = classify_one(row, FINAL_CONFIG)
        result["run_id"] = run_id

        stability_predictions = pd.concat([stability_predictions, pd.DataFrame([result])], ignore_index=True)
        stability_predictions = stability_predictions.drop_duplicates(subset=["pilot_id", "run_id"], keep="last").sort_values(["run_id", "pilot_id"])

        stability_predictions.to_csv(STABILITY_PATH, index=False, encoding="utf-8")

print("\nStability runs complete.")


=== Stability run 1/3 ===


  0%|          | 0/10 [00:00<?, ?it/s]


=== Stability run 2/3 ===


  0%|          | 0/10 [00:00<?, ?it/s]


=== Stability run 3/3 ===


  0%|          | 0/10 [00:00<?, ?it/s]


Stability runs complete.


In [ ]:
#INTER-RUN STABILITY

successful = stability_predictions[stability_predictions["error"].fillna("").eq("")].copy()
expected = len(development_set) * STABILITY_RUNS

if len(successful) != expected:
    raise ValueError(f"Expected {expected} successful predictions, but found {len(successful)}.")

successful["predicted_label_list"] = successful["predicted_labels"].apply(parse_label_list)

#sorting labels so order does not affect comparison
successful["label_set"] = successful["predicted_label_list"].apply(lambda x: tuple(sorted(x)))

#three predictions side-by-side
stability_table = successful.pivot(index="pilot_id", columns="run_id", values="label_set")
stability_table.columns = [f"run_{i}" for i in stability_table.columns]

#number of different predictions produced across the 3 runs
stability_table["n_unique_predictions"] = stability_table.apply(lambda row: len(set(row)), axis=1)
stability_table["all_3_identical"] = stability_table["n_unique_predictions"] == 1

display(stability_table)

stability_rate = stability_table["all_3_identical"].mean()

print(f"\nAll-three-runs exact stability: {stability_rate:.3f}")
print(f"Stable documents: {stability_table['all_3_identical'].sum()}/{len(stability_table)}")

,run_1,run_2,run_3,n_unique_predictions,all_3_identical
pilot_id,,,,,
1,"(Agricultural Land, Landscape, Visual and Heri...","(Agricultural Land, Landscape, Visual and Heri...","(Agricultural Land, Landscape, Visual and Heri...",1,True
9,"(Community and Economic Benefits, Consultation...","(Community and Economic Benefits, Consultation...","(Community and Economic Benefits, Consultation...",1,True
10,"(Cumulative Impact, Emergency Planning and Res...","(Cumulative Impact, Emergency Planning and Res...","(Cumulative Impact, Emergency Planning and Res...",1,True
13,"(Landscape, Visual and Heritage Impact, Site S...","(Landscape, Visual and Heritage Impact, Site S...","(Landscape, Visual and Heritage Impact, Site S...",1,True
16,"(Cumulative Impact, Emergency Planning and Res...","(Cumulative Impact, Emergency Planning and Res...","(Cumulative Impact, Emergency Planning and Res...",3,False
18,"(Consultation, Transparency and Information, C...","(Consultation, Transparency and Information, C...","(Community and Economic Benefits, Consultation...",3,False
24,"(Landscape, Visual and Heritage Impact,)","(Landscape, Visual and Heritage Impact,)","(Landscape, Visual and Heritage Impact,)",1,True
25,"(Landscape, Visual and Heritage Impact, Site S...","(Agricultural Land, Landscape, Visual and Heri...","(Agricultural Land, Landscape, Visual and Heri...",2,False
28,"(Emergency Planning and Response, Fire and Exp...","(Emergency Planning and Response, Fire and Exp...","(Emergency Planning and Response, Fire and Exp...",2,False



All-three-runs exact stability: 0.600
Stable documents: 6/10


In [ ]:
#pairwise exact agreement between runs
pairs = [("run_1", "run_2"), ("run_1", "run_3"), ("run_2", "run_3")]

for a, b in pairs:
    agreement = (stability_table[a] == stability_table[b]).mean()
    print(f"{a} vs {b}: {agreement:.3f}")

run_1 vs run_2: 0.700
run_1 vs run_3: 0.600
run_2 vs run_3: 0.700


In [ ]:
#checking pairwise label agreement between runs
pairs = [("run_1", "run_2"), ("run_1", "run_3"), ("run_2", "run_3")]

for a, b in pairs:
    labels_a = stability_table[a].apply(list)
    labels_b = stability_table[b].apply(list)

    y_a = mlb.transform(labels_a)
    y_b = mlb.transform(labels_b)

    micro_f1 = f1_score(y_a, y_b, average="micro", zero_division=0)

    print(f"{a} vs {b}: micro-F1 = {micro_f1:.3f}")

run_1 vs run_2: micro-F1 = 0.965
run_1 vs run_3: micro-F1 = 0.966
run_2 vs run_3: micro-F1 = 0.965


##Final terra low validation on the 20 held-out cases


In [ ]:
if VALIDATION_PREDICTIONS_PATH.exists():
    validation_predictions = pd.read_csv(VALIDATION_PREDICTIONS_PATH)
else:
    validation_predictions = pd.DataFrame()

completed = set()

if not validation_predictions.empty:
    successful = validation_predictions[validation_predictions["error"].fillna("").eq("")]
    completed = set(successful["pilot_id"])

jobs = [row for row in annotation_validation_set.to_dict("records") if row["pilot_id"] not in completed]

print(f"Terra API calls remaining: {len(jobs)}")

for row in tqdm(jobs):
    result = classify_one(row, FINAL_CONFIG)

    validation_predictions = pd.concat([validation_predictions, pd.DataFrame([result])], ignore_index=True)

    validation_predictions = validation_predictions.drop_duplicates(subset=["pilot_id"], keep="last").sort_values("pilot_id")

    # Save after every call
    validation_predictions.to_csv(VALIDATION_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

errors = validation_predictions[~validation_predictions["error"].fillna("").eq("")]

print(f"Saved: {VALIDATION_PREDICTIONS_PATH}")
print(f"Successful: {len(validation_predictions) - len(errors)}")
print(f"Failed: {len(errors)}")

if len(errors):
    display(errors[["pilot_id", "document_id", "error"]])

Terra API calls remaining: 20


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_annotation_validation_terra_low.csv
Successful: 20
Failed: 0


In [ ]:
#evaluation
successful = validation_predictions[validation_predictions["error"].fillna("").eq("")].copy()

if len(successful) != ANNOTATION_VALIDATION_SIZE:
    raise ValueError(f"Expected {ANNOTATION_VALIDATION_SIZE} successful predictions, found {len(successful)}.")

successful["predicted_label_list"] = successful["predicted_labels"].apply(parse_label_list)

validation_comparison = successful.merge(
    annotation_validation_set[["pilot_id", "document_id", "final_masked_text", "gold_labels"]],
    on=["pilot_id", "document_id"],
    how="inner",
    validate="one_to_one")

y_true = mlb.transform(validation_comparison["gold_labels"])
y_pred = mlb.transform(validation_comparison["predicted_label_list"])

validation_metrics = pd.DataFrame([{
    "config": "terra_low",
    "n": len(validation_comparison),
    "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
    "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
    "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    "sample_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
    "exact_match": accuracy_score(y_true, y_pred)}])

display(validation_metrics.style.format({
    "micro_precision": "{:.3f}",
    "micro_recall": "{:.3f}",
    "micro_f1": "{:.3f}",
    "sample_f1": "{:.3f}",
    "exact_match": "{:.3f}"}))

,config,n,micro_precision,micro_recall,micro_f1,sample_f1,exact_match
0,terra_low,20,0.963,0.977,0.970,0.979,0.700


In [ ]:
def get_label_differences(row):

    gold = set(row["gold_labels"])
    predicted = set(row["predicted_label_list"])

    return pd.Series({"missing_labels": sorted(gold - predicted), "extra_labels": sorted(predicted - gold)})


validation_details = validation_comparison.join(validation_comparison.apply(get_label_differences, axis=1))


validation_errors = validation_details[
    validation_details["missing_labels"].map(bool)
    |
    validation_details["extra_labels"].map(bool)]
     [[
        "pilot_id",
        "gold_labels",
        "predicted_label_list",
        "missing_labels",
        "extra_labels",
        "final_masked_text"]]


display(validation_errors)

,pilot_id,gold_labels,predicted_label_list,missing_labels,extra_labels,final_masked_text
2,4,"[Wildlife and Ecology, Fire and Explosion Risk...","[Wildlife and Ecology, Fire and Explosion Risk...",[],[Emergency Planning and Response],I would like to object to the proposed plan to...
6,8,"[Traffic, Fire and Explosion Risk, Landscape, ...","[Traffic, Fire and Explosion Risk, Landscape, ...",[Emergency Planning and Response],[Grid Connection and Electrical Infrastructure],Please find below a note of my strong objectio...
7,11,"[Wildlife and Ecology, Fire and Explosion Risk...","[Fire and Explosion Risk, Landscape, Visual an...",[Wildlife and Ecology],[],This formal objection is hereby lodged in resp...
9,14,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[],[Grid Connection and Electrical Infrastructure],The [PLACE] Battery Energy Storage System (BES...
12,19,"[Wildlife and Ecology, Traffic, Fire and Explo...","[Wildlife and Ecology, Traffic, Fire and Explo...",[],"[Consultation, Transparency and Information]",I would like to submit my objection to the abo...
14,21,"[Traffic, Fire and Explosion Risk, Emergency P...","[Traffic, Fire and Explosion Risk, Emergency P...",[Water and Soil Contamination],[Project Need],[APPLICATION_CODE] What is the battery storage...
